# Project: BBLF AI Selector v2
# Section: Optimisation & Simulation Tool
# Sub Section: Pre Tournament

Goal: Select the optimal starting 12 players prior to the start of the tournament 

Things to add:
1. Additional constraints to capture fantasy nuances e.g. captain points, vice captain trick 


# 0. Prerequistes

In [1]:
# pip install --upgrade pip

In [2]:
# !pip install mip==1.16rc0
!pip install mip

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# 0. Prerequistes

import pandas as pd
import numpy as np
import os
import random
from mip import Model, xsum, maximize, BINARY 
from scipy.stats import norm
import joblib
from concurrent.futures import ProcessPoolExecutor
import warnings
warnings.filterwarnings("ignore")

# Import intermediary functions
from Optimisation_Functions import roll_rnd_price_fn, optimise_fn_efp, optimise_fn_sim_fp

# Set working directory
os.getcwd()
directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2'
add_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/pre_tourny/'
over_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/add_data_created/overall'
py_data_directory = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/python_data/'
mock_data_dir = 'C:/Users/dilan/OneDrive/Documents/Data Science Projects/Repos/Big-Bash-Fantasy-AI-v2/data/mock_data/'

In [4]:
# Optimisation Strategy
opt_strat = 'efp' # Options: 'efp' (new player expected fantasy points appraoch), 'sim' (new player fantasy point distribution approach)
current_rnd = 1 # Current Round of the Tournament
exp_pts_model = 'all' # Options: 'bat_bowl' (batting + bowling model), 'all' (combined model)

# Constraints
squad_players = 15

# For Simulation Optimisation Strategy
conf_int = 0.2
sim_num = 2000

# 1. New EFP Approach

# a. Data Extraction 

In [5]:
# New Season Optimisation Strategy (Expected Fantasy Points)
if opt_strat == 'efp':
    
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Create Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture_upd.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df , team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected points csv file
    if exp_pts_model == 'bat_bowl':
        efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_bat_bowl_model_score_pre_tourny_short_upd.csv'), low_memory=False)
    elif exp_pts_model == 'all':
        efp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short_upd.csv'), low_memory=False)

    # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_points"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 11.68, player_df_raw["exp_pts"] + 4.42)
    player_df_raw["weight"] = 1

    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

    # Availablity Check: Switch to 0 if the team does not play in the round
    # BBL15 Byes: Rnd 4 - Stars, Rnd 5 - Sixers, Rnd 6 - Hurricanes, Rnd 7 - Strikers, Rnd 8 - Sixers, Rnd 9 - Hurricanes
    
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# b. Player price prediction for each round & data prep for optimisation

In [6]:
if opt_strat == 'efp':
    # Import Pricing Models
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Create rolling round player price dataframes
    player_df_r1, player_df_r2, player_df_r3, player_df_r4, player_df_r5, player_df_r6, player_df_r7, player_df_r8, player_df_r9 = roll_rnd_price_fn(player_df_raw, price_df, current_rnd, price_model_obj_1, price_model_obj_2, price_model_obj_3)

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

# c. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 12 + x bench player
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total budget of team is less than $1,802,500 (As bench costs $197,500)

## Optimisation Setup

In [7]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # a. EFP Optimisation Variables Setup
    # Round 1
    points_r1 = player_df_r1["exp_rnd_points"]
    price_r1 = player_df_r1["Price"]
    weight_r1 = player_df_r1["weight"]
    in_team_r1 = player_df_r1["In_Team"]
    available_r1 = player_df_r1["Available"]
    wk_weight_r1 = player_df_r1["Wk_f"]
    bat_weight_r1 = player_df_r1["Bat_f"]
    bowl_weight_r1 = player_df_r1["Bowl_f"]
    cnt_r1, max_player_r1 = squad_players, range(len(price_r1))
    play_cnt_r1, total_player_r1 = 12, range(len(price_r1))
    wk_cnt_r1, total_wk_r1 = 1, range(len(price_r1))
    bat_cnt_r1, total_bat_r1 = 6, range(len(price_r1))
    bowl_cnt_r1, total_bowl_r1 = 5, range(len(price_r1))
    budget_r1, total_budget_r1 = 1802500, range(len(price_r1))

    # Round 2
    points_r2 = player_df_r2["exp_rnd_points"]
    price_r2 = player_df_r2["Price"]
    weight_r2 = player_df_r2["weight"]
    in_team_r2 = player_df_r2["In_Team"]
    available_r2 = player_df_r2["Available"] 
    wk_weight_r2 = player_df_r2["Wk_f"]
    bat_weight_r2 = player_df_r2["Bat_f"]
    bowl_weight_r2 = player_df_r2["Bowl_f"]
    cnt_r2, max_player_r2 = squad_players, range(len(price_r2))
    play_cnt_r2, total_player_r2 = 12, range(len(price_r2))
    wk_cnt_r2, total_wk_r2 = 1, range(len(price_r2))
    bat_cnt_r2, total_bat_r2 = 6, range(len(price_r2))
    bowl_cnt_r2, total_bowl_r2 = 5, range(len(price_r2))
    budget_r2, total_budget_r2 = 1802500, range(len(price_r2))
    team_play_cnt_r2, total_team_player_r2 = (squad_players - 3), range(len(price_r2)) # At least 10 players from round 1 to be in round 2 team

    # Round 3
    points_r3 = player_df_r3["exp_rnd_points"]
    price_r3 = player_df_r3["Price"]
    weight_r3 = player_df_r3["weight"]
    in_team_r3 = player_df_r3["In_Team"]
    available_r3 = player_df_r3["Available"]
    wk_weight_r3 = player_df_r3["Wk_f"]
    bat_weight_r3 = player_df_r3["Bat_f"]
    bowl_weight_r3 = player_df_r3["Bowl_f"]
    cnt_r3, max_player_r3 = squad_players, range(len(price_r3))
    play_cnt_r3, total_player_r3 = 12, range(len(price_r3))
    wk_cnt_r3, total_wk_r3 = 1, range(len(price_r3))
    bat_cnt_r3, total_bat_r3 = 6, range(len(price_r3))
    bowl_cnt_r3, total_bowl_r3 = 5, range(len(price_r3))
    budget_r3, total_budget_r3 = 1802500, range(len(price_r3))
    team_play_cnt_r3, total_team_player_r3 = (squad_players - 3), range(len(price_r3)) # At least 10 players from round 2 to be in round 3 team

    # Round 4
    points_r4 = player_df_r4["exp_rnd_points"]
    price_r4 = player_df_r4["Price"]
    weight_r4 = player_df_r4["weight"]
    in_team_r4 = player_df_r4["In_Team"]
    available_r4 = player_df_r4["Available"]
    wk_weight_r4 = player_df_r4["Wk_f"]
    bat_weight_r4 = player_df_r4["Bat_f"]
    bowl_weight_r4 = player_df_r4["Bowl_f"]
    cnt_r4, max_player_r4 = squad_players, range(len(price_r4))
    play_cnt_r4, total_player_r4 = 12, range(len(price_r4))
    wk_cnt_r4, total_wk_r4 = 1, range(len(price_r4))
    bat_cnt_r4, total_bat_r4 = 6, range(len(price_r4))
    bowl_cnt_r4, total_bowl_r4 = 5, range(len(price_r4))
    budget_r4, total_budget_r4 = 1802500, range(len(price_r4))
    team_play_cnt_r4, total_team_player_r4 = (squad_players - 3), range(len(price_r4)) # At least 10 players from round 3 to be in round 4 team

    # Round 5
    points_r5 = player_df_r5["exp_rnd_points"]
    price_r5 = player_df_r5["Price"]
    weight_r5 = player_df_r5["weight"]
    in_team_r5 = player_df_r5["In_Team"]
    available_r5 = player_df_r5["Available"]
    wk_weight_r5 = player_df_r5["Wk_f"]
    bat_weight_r5 = player_df_r5["Bat_f"]
    bowl_weight_r5 = player_df_r5["Bowl_f"]
    cnt_r5, max_player_r5 = squad_players, range(len(price_r5))
    play_cnt_r5, total_player_r5 = 12, range(len(price_r5))
    wk_cnt_r5, total_wk_r5 = 1, range(len(price_r5))
    bat_cnt_r5, total_bat_r5 = 6, range(len(price_r5))
    bowl_cnt_r5, total_bowl_r5 = 5, range(len(price_r5))
    budget_r5, total_budget_r5 = 1802500, range(len(price_r5))
    team_play_cnt_r5, total_team_player_r5 = (squad_players - 3), range(len(price_r5)) # At least 10 players from round 4 to be in round 5 team

    # Round 6
    points_r6 = player_df_r6["exp_rnd_points"]
    price_r6 = player_df_r6["Price"]
    weight_r6 = player_df_r6["weight"]
    in_team_r6 = player_df_r6["In_Team"]
    available_r6 = player_df_r6["Available"]
    wk_weight_r6 = player_df_r6["Wk_f"]
    bat_weight_r6 = player_df_r6["Bat_f"]
    bowl_weight_r6 = player_df_r6["Bowl_f"]
    cnt_r6, max_player_r6 = squad_players, range(len(price_r6))
    play_cnt_r6, total_player_r6 = 12, range(len(price_r6))
    wk_cnt_r6, total_wk_r6 = 1, range(len(price_r6))
    bat_cnt_r6, total_bat_r6 = 6, range(len(price_r6))
    bowl_cnt_r6, total_bowl_r6 = 5, range(len(price_r6))
    budget_r6, total_budget_r6 = 1783500, range(len(price_r6))
    team_play_cnt_r6, total_team_player_r6 = (squad_players - 3), range(len(price_r6)) # At least 10 players from round 5 to be in round 6 team

    # Round 7
    points_r7 = player_df_r7["exp_rnd_points"]
    price_r7 = player_df_r7["Price"]
    weight_r7 = player_df_r7["weight"]
    in_team_r7 = player_df_r7["In_Team"]
    available_r7 = player_df_r7["Available"]
    wk_weight_r7 = player_df_r7["Wk_f"]
    bat_weight_r7 = player_df_r7["Bat_f"]
    bowl_weight_r7 = player_df_r7["Bowl_f"]
    cnt_r7, max_player_r7 = squad_players, range(len(price_r7))
    play_cnt_r7, total_player_r7 = 12, range(len(price_r7))
    wk_cnt_r7, total_wk_r7 = 1, range(len(price_r7))
    bat_cnt_r7, total_bat_r7 = 6, range(len(price_r7))
    bowl_cnt_r7, total_bowl_r7 = 5, range(len(price_r7))
    budget_r7, total_budget_r7 = 1802500, range(len(price_r7))
    team_play_cnt_r7, total_team_player_r7 = (squad_players - 3), range(len(price_r7)) # At least 10 players from round 6 to be in round 7 team

    # Round 8
    points_r8 = player_df_r8["exp_rnd_points"]
    price_r8 = player_df_r8["Price"]
    weight_r8 = player_df_r8["weight"]
    in_team_r8 = player_df_r8["In_Team"]
    available_r8 = player_df_r8["Available"]
    wk_weight_r8 = player_df_r8["Wk_f"]
    bat_weight_r8 = player_df_r8["Bat_f"]
    bowl_weight_r8 = player_df_r8["Bowl_f"]
    cnt_r8, max_player_r8 = squad_players, range(len(price_r8))
    play_cnt_r8, total_player_r8 = 12, range(len(price_r8))
    wk_cnt_r8, total_wk_r8 = 1, range(len(price_r8))
    bat_cnt_r8, total_bat_r8 = 6, range(len(price_r8))
    bowl_cnt_r8, total_bowl_r8 = 5, range(len(price_r8))
    budget_r8, total_budget_r8 = 1802500, range(len(price_r8))
    team_play_cnt_r8, total_team_player_r8 = (squad_players - 3), range(len(price_r8)) # At least 10 players from round 7 to be in round 8 team

    # Round 9
    points_r9 = player_df_r9["exp_rnd_points"]
    price_r9 = player_df_r9["Price"]
    weight_r9 = player_df_r9["weight"]
    in_team_r9 = player_df_r9["In_Team"]
    available_r9 = player_df_r9["Available"]
    wk_weight_r9 = player_df_r9["Wk_f"]
    bat_weight_r9 = player_df_r9["Bat_f"]
    bowl_weight_r9 = player_df_r9["Bowl_f"]
    cnt_r9, max_player_r9 = squad_players, range(len(price_r9))
    play_cnt_r9, total_player_r9 = 12, range(len(price_r9))
    wk_cnt_r9, total_wk_r9 = 1, range(len(price_r9))
    bat_cnt_r9, total_bat_r9 = 6, range(len(price_r9))
    bowl_cnt_r9, total_bowl_r9 = 5, range(len(price_r9))
    budget_r9, total_budget_r9 = 1802500, range(len(price_r9))
    team_play_cnt_r9, total_team_player_r9 = (squad_players - 3), range(len(price_r9)) # At least 10 players from round 8 to be in round 9 team

else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

## Optimisation Runner

In [8]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    sel_player_df, sel_player_df_r1, sel_player_df_r2, sel_player_df_r3, sel_player_df_r4, sel_player_df_r5, sel_player_df_r6, sel_player_df_r7,sel_player_df_r8, sel_player_df_r9 = optimise_fn_efp(
        # Round 1
        points_r1, price_r1, weight_r1, in_team_r1, available_r1, wk_weight_r1, bat_weight_r1, bowl_weight_r1,
        play_cnt_r1, total_player_r1, wk_cnt_r1, total_wk_r1, bat_cnt_r1, total_bat_r1, bowl_cnt_r1, total_bowl_r1,
        budget_r1, total_budget_r1, player_df_r1, cnt_r1, max_player_r1,
        # Round 2
        points_r2, price_r2, weight_r2, in_team_r2, available_r2, wk_weight_r2, bat_weight_r2, bowl_weight_r2,
        play_cnt_r2, total_player_r2, wk_cnt_r2, total_wk_r2, bat_cnt_r2, total_bat_r2, bowl_cnt_r2, total_bowl_r2,
        budget_r2, total_budget_r2, team_play_cnt_r2, total_team_player_r2, player_df_r2, cnt_r2, max_player_r2,
        # Round 3
        points_r3, price_r3, weight_r3, in_team_r3, available_r3, wk_weight_r3, bat_weight_r3, bowl_weight_r3,
        play_cnt_r3, total_player_r3, wk_cnt_r3, total_wk_r3, bat_cnt_r3, total_bat_r3, bowl_cnt_r3, total_bowl_r3,
        budget_r3, total_budget_r3, team_play_cnt_r3, total_team_player_r3, player_df_r3, cnt_r3, max_player_r3,
        # Round 4
        points_r4, price_r4, weight_r4, in_team_r4, available_r4, wk_weight_r4, bat_weight_r4, bowl_weight_r4,
        play_cnt_r4, total_player_r4, wk_cnt_r4, total_wk_r4, bat_cnt_r4, total_bat_r4, bowl_cnt_r4, total_bowl_r4,
        budget_r4, total_budget_r4, team_play_cnt_r4, total_team_player_r4, player_df_r4, cnt_r4, max_player_r4,
        # Round 5
        points_r5, price_r5, weight_r5, in_team_r5, available_r5, wk_weight_r5, bat_weight_r5, bowl_weight_r5,
        play_cnt_r5, total_player_r5, wk_cnt_r5, total_wk_r5, bat_cnt_r5, total_bat_r5, bowl_cnt_r5, total_bowl_r5,
        budget_r5, total_budget_r5, team_play_cnt_r5, total_team_player_r5, player_df_r5, cnt_r5, max_player_r5,
        # Round 6
        points_r6, price_r6, weight_r6, in_team_r6, available_r6, wk_weight_r6, bat_weight_r6, bowl_weight_r6,
        play_cnt_r6, total_player_r6, wk_cnt_r6, total_wk_r6, bat_cnt_r6, total_bat_r6, bowl_cnt_r6, total_bowl_r6,
        budget_r6, total_budget_r6, team_play_cnt_r6, total_team_player_r6, player_df_r6, cnt_r6, max_player_r6,
        # Round 7
        points_r7, price_r7, weight_r7, in_team_r7, available_r7, wk_weight_r7, bat_weight_r7, bowl_weight_r7,
        play_cnt_r7, total_player_r7, wk_cnt_r7, total_wk_r7, bat_cnt_r7, total_bat_r7, bowl_cnt_r7, total_bowl_r7,
        budget_r7, total_budget_r7, team_play_cnt_r7, total_team_player_r7, player_df_r7, cnt_r7, max_player_r7,
        # Round 8
        points_r8, price_r8, weight_r8, in_team_r8, available_r8, wk_weight_r8, bat_weight_r8, bowl_weight_r8,
        play_cnt_r8, total_player_r8, wk_cnt_r8, total_wk_r8, bat_cnt_r8, total_bat_r8, bowl_cnt_r8, total_bowl_r8,
        budget_r8, total_budget_r8, team_play_cnt_r8, total_team_player_r8, player_df_r8, cnt_r8, max_player_r8,
        # Round 9
        points_r9, price_r9, weight_r9, in_team_r9, available_r9, wk_weight_r9, bat_weight_r9, bowl_weight_r9,
        play_cnt_r9, total_player_r9, wk_cnt_r9, total_wk_r9, bat_cnt_r9, total_bat_r9, bowl_cnt_r9, total_bowl_r9,
        budget_r9, total_budget_r9, team_play_cnt_r9, total_team_player_r9, player_df_r9, cnt_r9, max_player_r9
    )

    # Save Optimal Team to CSV
    if exp_pts_model == 'all':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'), index=False)
        sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_rnd.csv'), index=False)
        print("EFP Optimal Team Saved")
    elif exp_pts_model == 'bat_bowl':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual.csv'), index=False)
        sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual_rnd.csv'), index=False)
        print("EFP Bat + Bowl Optimal Team Saved")
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

----- Optimal Team Selection Summary -----
----- Round 1 -----
Total Expected Points (rnd 1): 1081.50694
Total Team Cost (rnd 1): 1864100.0
Captain (rnd 1): Mitch Owen
Bench Player (rnd 1): Jack Wildermuth ($58,500.0)
Current Players Remaining (rnd 1): 0.0
----- Round 2 -----
Total Expected Points (rnd 2): 1034.46738
Total Team Cost (rnd 2): 1880769.0052594063
Captain (rnd 2): Daniel Sams
Bench Player (rnd 2): Ben McDermott ($85,931.27647272727)
Current Players Remaining (rnd 2): 12
----- Round 3 -----
Total Expected Points (rnd 3): 1141.3391780000002
Total Team Cost (rnd 3): 1954984.967424724
Captain (rnd 3): Mitch Owen
Bench Player (rnd 3): Jack Wildermuth ($80,852.54186651793)
Current Players Remaining (rnd 3): 12
----- Round 4 -----
Total Expected Points (rnd 4): 929.9859839999999
Total Team Cost (rnd 4): 2042705.6953309118
Captain (rnd 4): Cooper Connolly
Bench Player (rnd 4): Haris Rauf ($149,285.13029434468)
Current Players Remaining (rnd 4): 12
----- Round 5 -----
Total Expecte

## Optimisation Results

In [9]:
# EFP Whole Tournament Optimisation
if opt_strat == 'efp':
    # Display Round-wise Selected Teams
    print("Round 1 Selected Team")
    display(sel_player_df_r1.sort_values(by='exp_rnd_points', ascending=False))

    # Save Optimal Team to CSV
    if exp_pts_model == 'all':
        # sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team.csv'), index=False)
        # sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_rnd.csv'), index=False)
        print("EFP Optimal Team Saved")
    elif exp_pts_model == 'bat_bowl':
        sel_player_df.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual.csv'), index=False)
        sel_player_df_r1.to_csv(os.path.join(add_data_directory,'optim/EFP_optimal_pre_tourny_team_dual_rnd.csv'), index=False)
        print("EFP Bat + Bowl Optimal Team Saved")
else:
    print("Change opt_strat to efp for player efp Optimisation Strategy")

Round 1 Selected Team


,Name,Round,Price,Team,Wk_f,Bat_f,Bowl_f,Role,weight,Available,In_Team,exp_rnd_points,games_in_round,Is_Bench
5,Mitch Owen,1,192800.0,Hobart Hurricanes,0.0,1.0,1.0,ALLR,1.0,1.0,0.0,104.093112,2.0,0
1,Chris Jordan,1,105500.0,Hobart Hurricanes,0.0,0.0,0.0,BWL,1.0,1.0,0.0,102.026900,2.0,0
6,Nathan Ellis,1,153900.0,Hobart Hurricanes,0.0,0.0,0.0,BWL,1.0,1.0,0.0,99.379956,2.0,0
9,Riley Meredith,1,130200.0,Hobart Hurricanes,0.0,0.0,0.0,BWL,1.0,1.0,0.0,95.305530,2.0,0
4,Matthew Wade,1,94900.0,Hobart Hurricanes,1.0,1.0,0.0,WK,1.0,1.0,0.0,85.776768,2.0,0
0,Ben McDermott,1,79500.0,Hobart Hurricanes,1.0,1.0,0.0,WK,1.0,1.0,0.0,81.451938,2.0,0
7,Nikhil Chaudhary,1,126900.0,Hobart Hurricanes,0.0,1.0,1.0,ALLR,1.0,1.0,0.0,77.179440,2.0,0
8,Rehan Ahmed,1,117500.0,Hobart Hurricanes,0.0,1.0,1.0,ALLR,1.0,1.0,0.0,77.155568,2.0,0
10,Rishad Hossain,1,117500.0,Hobart Hurricanes,0.0,1.0,1.0,ALLR,1.0,1.0,0.0,77.155568,2.0,0
3,Matthew Short,1,214300.0,Adelaide Strikers,0.0,1.0,1.0,ALLR,1.0,1.0,0.0,66.217314,1.0,0


EFP Optimal Team Saved


# 2. New Distribution Approach

# a. Data Extraction 

In [114]:
# New Optimisation Strategy (Player Fantasy Point Distribution)
if opt_strat == 'sim':
    # Data Extraction 
    # Pull in player price data csv file 
    price_df = pd.read_csv(os.path.join(add_data_directory,'player_price_2026.csv'), low_memory=False)

    # Player Role Flags
    price_df["Wk_f"] = np.where((price_df["Role"] == "WK"), 1, 0)
    price_df["Bat_f"] = np.where((price_df["Role"] == "WK") |(price_df["Role"] == "BAT") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df["Bowl_f"] = np.where((price_df["Role"] == "BOWL") | (price_df["Role"] == "ALLR") , 1, 0)
    price_df = price_df[["Full_Name", "Price", "Team","Wk_f", "Bat_f", "Bowl_f", "Role", "Available", "In_Team"]].rename(columns = {"Full_Name": "Name"}) 

    team_fix_df = pd.read_csv(os.path.join(over_data_directory,'team_loc_fixture_upd.csv'), low_memory=False)
    team_fix_df = team_fix_df[team_fix_df.Round >= current_rnd].dropna()

    # Join team fixture to player price to create a row for each game
    game_df = pd.merge(price_df, team_fix_df, left_on = ["Team"], right_on = ["Team"], how = "left")

    # Pull player expected & std points csv file
    if exp_pts_model == 'bat_bowl':
        efp_raw_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_bat_bowl_model_score_pre_tourny_short_upd.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_sfp_model_score_pre_tourny_short_upd.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    elif exp_pts_model == 'all':
        efp_raw_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_efp_model_score_pre_tourny_short_upd.csv'), low_memory=False)
        sfp_df = pd.read_csv(os.path.join(py_data_directory,'bbl15_sfp_model_score_pre_tourny_short_upd.csv'), low_memory=False)
        efp_df = efp_raw_df.merge(sfp_df, left_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], right_on = ["player", "Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")

    # # Join on expected points for each game
    player_df_raw = pd.merge(game_df, efp_df, left_on = ["Name", "Team", "Opposition", "Venue", "Home_f","Round"], right_on = ["Name", "Team", "opp", "venue", "Home_f","Round"], how = "left")
    player_df_raw["exp_pts"] = np.where((player_df_raw["Role"] == "WK"), player_df_raw["exp_pts"] + 11.68, player_df_raw["exp_pts"] + 4.42)
    player_df_raw = player_df_raw.rename(columns={"exp_pts": "mean"})
    player_df_raw = player_df_raw.rename(columns={"sd_pts": "std_dev"})

    player_df_raw["weight"] = 1
    
    # Create game count per player
    player_df_raw["game_num"] = player_df_raw.groupby("Name").cumcount() + 1

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


# b. Optimisation Process (Pre Tournament Selection) - Starting 12 Players

Optimisation Objective: Maximise the number of expected fantasy points

Constraints: 
1. Number of players selected must be 13 (Incl Bench Player)
2. Atleast 1 wicketkeeper
3. Atleast 6 batters
4. Atleast 5 bowlers
5. Total starting budget of team is less than $1,802,500 (As bench costs $197,500)

In [115]:
# Sim FP Whole Tournament Optimisation
if opt_strat == 'sim':
    # Load Model Objects
    price_model_obj_1 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_1_game'))
    price_model_obj_2 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_2_game'))
    price_model_obj_3 = joblib.load(os.path.join(directory, 'python_script/pricing-models/models/bbl15_price_model_3_game'))

    # Run Simulation Optimisation Function (with parallel execution enabled)
    all_sim_sel_players = optimise_fn_sim_fp(
        conf_int, sim_num, current_rnd, player_df_raw, price_df, 
        price_model_obj_1, price_model_obj_2, price_model_obj_3,
        squad_players, use_parallel=True  # Set to False for sequential execution
    )

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")
 

For 20% simulated points confidence interval the lower z score is -0.253 and upper z score is 0.253
Completed 10/2000 simulations
Completed 20/2000 simulations
Completed 30/2000 simulations
Completed 40/2000 simulations
Completed 50/2000 simulations
Completed 60/2000 simulations
Completed 70/2000 simulations
Completed 80/2000 simulations
Completed 90/2000 simulations
Completed 100/2000 simulations
Completed 110/2000 simulations
Completed 120/2000 simulations
Completed 130/2000 simulations
Completed 140/2000 simulations
Completed 150/2000 simulations
Completed 160/2000 simulations
Completed 170/2000 simulations
Completed 180/2000 simulations
Completed 190/2000 simulations
Completed 200/2000 simulations
Completed 210/2000 simulations
Completed 220/2000 simulations
Completed 230/2000 simulations
Completed 240/2000 simulations
Completed 250/2000 simulations
Completed 260/2000 simulations
Completed 270/2000 simulations
Completed 280/2000 simulations
Completed 290/2000 simulations
Completed 

In [116]:
# Find Most Common Player Combinations Across Simulations
if opt_strat == 'sim':
    from collections import Counter
    
    # Group by simulation and create player combination sets
    sim_combinations = []
    
    for sim_id in all_sim_sel_players['Simulation'].unique():
        sim_players = all_sim_sel_players[all_sim_sel_players['Simulation'] == sim_id]['Name'].unique()
        # Create a frozenset to make it hashable for counting
        player_set = frozenset(sim_players)
        sim_combinations.append(player_set)
    
    # Count frequency of each combination
    combination_counts = Counter(sim_combinations)
    
    # Sort by frequency (descending)
    sorted_combinations = sorted(combination_counts.items(), key=lambda x: x[1], reverse=True)
    
    # Display results
    print(f"Total Unique Player Combinations: {len(combination_counts)}")
    print(f"Total Simulations: {len(sim_combinations)}\n")
    
    # Show top 10 combinations
    print("=" * 80)
    print("TOP 10 MOST COMMON PLAYER COMBINATIONS")
    print("=" * 80)
    
    for rank, (player_combo, count) in enumerate(sorted_combinations[:10], 1):
        percentage = (count / len(sim_combinations)) * 100
        print(f"\nRank {rank}: {count}/{len(sim_combinations)} simulations ({percentage:.1f}%)")
        print(f"Players: {sorted(list(player_combo))}")
        print("-" * 80)

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")

Total Unique Player Combinations: 966
Total Simulations: 2000

TOP 10 MOST COMMON PLAYER COMBINATIONS

Rank 1: 18/2000 simulations (0.9%)
Players: ['Ben McDermott', 'Chris Jordan', 'Daniel Sams', 'Jamie Overton', 'Matthew Short', 'Matthew Wade', 'Mitch Owen', 'Nathan Ellis', 'Nathan McAndrew', 'Riley Meredith', 'Rishad Hossain', 'Tim David']
--------------------------------------------------------------------------------

Rank 2: 18/2000 simulations (0.9%)
Players: ['Ben McDermott', 'Chris Jordan', 'Daniel Sams', 'Matthew Short', 'Matthew Wade', 'Mitch Owen', 'Nathan Ellis', 'Nathan McAndrew', 'Rehan Ahmed', 'Riley Meredith', 'Rishad Hossain', 'Will Sutherland']
--------------------------------------------------------------------------------

Rank 3: 17/2000 simulations (0.9%)
Players: ['Ben McDermott', 'Chris Jordan', 'Daniel Sams', 'Jack Wildermuth', 'Jamie Overton', 'Matthew Short', 'Matthew Wade', 'Mitch Owen', 'Nathan Ellis', 'Nikhil Chaudhary', 'Rehan Ahmed', 'Riley Meredith']
--

In [117]:
# Create DataFrame Summary of Most Common Combinations
if opt_strat == 'sim':
    # Create a dataframe with combination rankings
    combination_summary = []
    
    for rank, (player_combo, count) in enumerate(sorted_combinations, 1):
        percentage = (count / len(sim_combinations)) * 100
        combination_summary.append({
            'Rank': rank,
            'Frequency': count,
            'Percentage': f"{percentage:.1f}%",
            'Players': ', '.join(sorted(list(player_combo))),
            'Num_Players': len(player_combo)
        })
    
    combination_summary_df = pd.DataFrame(combination_summary)
    
    # Display the top combinations as a nice table
    print("\nCOMBINATION SUMMARY TABLE:")
    print(combination_summary_df.head(10).to_string(index=False))
    
    # Get the most common combination
    if sorted_combinations:
        most_common_players = sorted(list(sorted_combinations[0][0]))
        most_common_count = sorted_combinations[0][1]
        most_common_percentage = (most_common_count / len(sim_combinations)) * 100
        
        print(f"\n{'='*80}")
        print(f"MOST COMMON COMBINATION SELECTED:")
        print(f"{'='*80}")
        print(f"Appears in: {most_common_count}/{len(sim_combinations)} simulations ({most_common_percentage:.1f}%)")
        print(f"\nPlayers ({len(most_common_players)}):")
        for i, player in enumerate(most_common_players, 1):
            print(f"  {i:2d}. {player}")
        print(f"{'='*80}")

    # Format most common players into dataframe
    most_common_players = pd.DataFrame(most_common_players, columns=['Name'])

    # Merge with player details
    most_common_players = pd.merge(most_common_players, price_df, on='Name', how='left')
    
    # Save Combination Summary to CSV
    if exp_pts_model == 'all':
        most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_team_12.csv'), index=False)
        combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_combos_12.csv'), index=False)

        print("Sim FP Combination Summary Saved")
    elif exp_pts_model == 'bat_bowl':
        most_common_players.to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_team_dual.csv'), index=False)
        combination_summary_df.head(5).to_csv(os.path.join(add_data_directory,'optim/SFP_optimal_pre_tourny_combos.csv'), index=False)

        print("Sim FP Combination Summary Saved")

else:
    print("Change opt_strat to sim for player fp distribution Optimisation Strategy")


COMBINATION SUMMARY TABLE:
 Rank  Frequency Percentage                                                                                                                                                                            Players  Num_Players
    1         18       0.9%         Ben McDermott, Chris Jordan, Daniel Sams, Jamie Overton, Matthew Short, Matthew Wade, Mitch Owen, Nathan Ellis, Nathan McAndrew, Riley Meredith, Rishad Hossain, Tim David           12
    2         18       0.9%     Ben McDermott, Chris Jordan, Daniel Sams, Matthew Short, Matthew Wade, Mitch Owen, Nathan Ellis, Nathan McAndrew, Rehan Ahmed, Riley Meredith, Rishad Hossain, Will Sutherland           12
    3         17       0.9%     Ben McDermott, Chris Jordan, Daniel Sams, Jack Wildermuth, Jamie Overton, Matthew Short, Matthew Wade, Mitch Owen, Nathan Ellis, Nikhil Chaudhary, Rehan Ahmed, Riley Meredith           12
    4         17       0.9% Ben McDermott, Chris Jordan, Jack Wildermuth, Matthew Short, Mat